# Module 0: OLS Fundamentals in Matrix Form

## Learning Objectives

By completing this notebook, you will:
1. Implement OLS estimation using matrix algebra from scratch
2. Understand the geometric interpretation of OLS as projection
3. Compute and interpret standard errors and confidence intervals
4. Verify OLS properties: residuals orthogonal to predictors, sum to zero
5. Compare manual implementation to library functions (statsmodels)

## Prerequisites

- Linear algebra: Matrix multiplication, transpose, inverse
- Python: NumPy arrays, basic pandas
- Statistics: Expected value, variance, hypothesis testing

## Estimated Time

60-75 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Setup complete")

## 1. The OLS Estimator: Theory Recap

### The Problem

Given data $(y, X)$ where:
- $y$ is an $n \times 1$ vector of outcomes
- $X$ is an $n \times k$ matrix of predictors (including intercept column)

We want to find coefficients $\beta$ that minimize the sum of squared residuals:

$$\min_{\beta} \sum_{i=1}^n (y_i - X_i\beta)^2 = \min_{\beta} (y - X\beta)'(y - X\beta)$$

### The Solution

The OLS estimator is:

$$\hat{\beta} = (X'X)^{-1}X'y$$

This comes from the first-order condition: $X'X\beta = X'y$ (the **normal equations**).

### Key Properties

1. **Residuals orthogonal to predictors:** $X'\hat{\epsilon} = 0$
2. **Residuals sum to zero** (if intercept included): $\sum \hat{\epsilon}_i = 0$
3. **Variance formula:** $\text{Var}(\hat{\beta}) = \sigma^2(X'X)^{-1}$

## 2. Implementing OLS from Scratch

In [ ]:
def ols_from_scratch(y, X, add_intercept=True):
    """
    Estimate OLS using matrix algebra.
    
    Parameters
    ----------
    y : array-like, shape (n,)
        Dependent variable
    X : array-like, shape (n, k) or (n,)
        Independent variable(s)
    add_intercept : bool
        Whether to add constant term
    
    Returns
    -------
    results : dict
        Dictionary with estimation results:
        - beta_hat: Coefficient estimates
        - y_hat: Fitted values
        - residuals: Residuals
        - sigma2: Residual variance
        - se_beta: Standard errors
        - t_stats: t-statistics
        - p_values: p-values for two-sided tests
        - r_squared: R-squared
    """
    # Convert to numpy arrays and handle dimensions
    y = np.asarray(y).flatten()
    X = np.asarray(X)
    
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    
    # Add intercept if requested
    if add_intercept:
        X = np.column_stack([np.ones(len(y)), X])
    
    n, k = X.shape
    
    # Compute OLS estimate: beta_hat = (X'X)^{-1}X'y
    XtX = X.T @ X
    Xty = X.T @ y
    
    # Use solve instead of inv for numerical stability
    beta_hat = np.linalg.solve(XtX, Xty)
    
    # Fitted values: y_hat = X * beta_hat
    y_hat = X @ beta_hat
    
    # Residuals: e = y - y_hat
    residuals = y - y_hat
    
    # Sum of squared residuals
    SSR = residuals @ residuals
    
    # Total sum of squares
    TSS = np.sum((y - y.mean())**2)
    
    # R-squared
    r_squared = 1 - SSR / TSS
    
    # Residual variance: sigma^2 = SSR / (n - k)
    sigma2 = SSR / (n - k)
    
    # Variance-covariance matrix: Var(beta_hat) = sigma^2 * (X'X)^{-1}
    XtX_inv = np.linalg.inv(XtX)
    var_beta = sigma2 * XtX_inv
    
    # Standard errors
    se_beta = np.sqrt(np.diag(var_beta))
    
    # t-statistics: t = beta_hat / se(beta_hat)
    t_stats = beta_hat / se_beta
    
    # p-values (two-sided test)
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))
    
    return {
        'beta_hat': beta_hat,
        'y_hat': y_hat,
        'residuals': residuals,
        'sigma2': sigma2,
        'se_beta': se_beta,
        't_stats': t_stats,
        'p_values': p_values,
        'r_squared': r_squared,
        'n': n,
        'k': k
    }

print("✅ OLS function defined")

### Generate Synthetic Data

Let's create data from a known data-generating process (DGP):

$$y_i = 2 + 1.5 x_{1i} - 0.8 x_{2i} + \epsilon_i$$

where $\epsilon_i \sim N(0, 1)$.

In [ ]:
# Sample size
n = 200

# Generate predictors
X1 = np.random.randn(n)
X2 = np.random.randn(n)
X = np.column_stack([X1, X2])

# True parameters
beta_true = np.array([2.0, 1.5, -0.8])  # [intercept, beta1, beta2]

# Generate outcome
epsilon = np.random.randn(n)
y = beta_true[0] + X @ beta_true[1:] + epsilon

print(f"Generated {n} observations")
print(f"True parameters: {beta_true}")

### Estimate the Model

In [ ]:
# Estimate OLS
results = ols_from_scratch(y, X, add_intercept=True)

# Display results
print("OLS Estimation Results")
print("=" * 70)
print(f"{'Variable':<15} {'Coef':>10} {'Std Err':>10} {'t':>8} {'P>|t|':>10}")
print("=" * 70)

var_names = ['Intercept', 'X1', 'X2']
for i, var in enumerate(var_names):
    print(f"{var:<15} {results['beta_hat'][i]:>10.4f} {results['se_beta'][i]:>10.4f} "
          f"{results['t_stats'][i]:>8.3f} {results['p_values'][i]:>10.4f}")

print("=" * 70)
print(f"R-squared: {results['r_squared']:.4f}")
print(f"Residual std error: {np.sqrt(results['sigma2']):.4f}")
print(f"Degrees of freedom: {results['n'] - results['k']}")

# Compare to true values
print("\nComparison to True Values:")
print(f"{'Variable':<15} {'True':>10} {'Estimated':>10} {'Difference':>10}")
print("-" * 50)
for i, var in enumerate(var_names):
    diff = results['beta_hat'][i] - beta_true[i]
    print(f"{var:<15} {beta_true[i]:>10.4f} {results['beta_hat'][i]:>10.4f} {diff:>10.4f}")

## 3. Verifying OLS Properties

Let's verify that our implementation satisfies the theoretical properties of OLS.

### Property 1: Residuals are Orthogonal to Predictors

The normal equations $X'X\beta = X'y$ imply that $X'\hat{\epsilon} = 0$.

This means the residuals are uncorrelated with each predictor (including the intercept).

In [ ]:
# Add intercept to X for this check
X_with_intercept = np.column_stack([np.ones(n), X])

# Compute X' * residuals
orthogonality_check = X_with_intercept.T @ results['residuals']

print("Orthogonality Check (X' * residuals):")
print(orthogonality_check)
print("\nAll values should be close to zero (within numerical precision).")
print(f"Maximum absolute value: {np.max(np.abs(orthogonality_check)):.2e}")

# Verify
assert np.allclose(orthogonality_check, 0, atol=1e-10), "Orthogonality property violated!"
print("✅ Orthogonality property satisfied")

### Property 2: Residuals Sum to Zero

When an intercept is included, $\sum_{i=1}^n \hat{\epsilon}_i = 0$.

This follows from the orthogonality of residuals with the intercept (constant) column.

In [ ]:
residual_sum = np.sum(results['residuals'])

print(f"Sum of residuals: {residual_sum:.2e}")
print("Should be close to zero.")

assert np.abs(residual_sum) < 1e-10, "Residuals do not sum to zero!"
print("✅ Residual sum property satisfied")

### Property 3: Projection Matrix Properties

The projection matrix $P = X(X'X)^{-1}X'$ has two key properties:
1. **Symmetric:** $P = P'$
2. **Idempotent:** $P^2 = P$

Similarly for the residual maker $M = I - P$.

In [ ]:
# Compute projection matrix
XtX_inv = np.linalg.inv(X_with_intercept.T @ X_with_intercept)
P = X_with_intercept @ XtX_inv @ X_with_intercept.T

# Check symmetry: P = P'
is_symmetric = np.allclose(P, P.T)
print(f"P is symmetric: {is_symmetric}")

# Check idempotence: P^2 = P
is_idempotent = np.allclose(P @ P, P)
print(f"P is idempotent: {is_idempotent}")

# Check that P * y = y_hat
y_hat_from_P = P @ y
matches_fitted = np.allclose(y_hat_from_P, results['y_hat'])
print(f"P * y = y_hat: {matches_fitted}")

# Check residual maker M = I - P
M = np.eye(n) - P
residuals_from_M = M @ y
matches_residuals = np.allclose(residuals_from_M, results['residuals'])
print(f"M * y = residuals: {matches_residuals}")

print("\n✅ All projection matrix properties verified")

## 4. Geometric Interpretation: Visualization

For the simple regression case (one predictor), we can visualize OLS as finding the best-fit line.

In [ ]:
# Simple regression example
n_simple = 50
X_simple = np.random.randn(n_simple) * 2
y_simple = 1 + 2 * X_simple + np.random.randn(n_simple)

# Estimate
results_simple = ols_from_scratch(y_simple, X_simple)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot
ax.scatter(X_simple, y_simple, alpha=0.6, s=50, label='Data')

# Fitted line
X_line = np.linspace(X_simple.min(), X_simple.max(), 100)
y_line = results_simple['beta_hat'][0] + results_simple['beta_hat'][1] * X_line
ax.plot(X_line, y_line, 'r-', linewidth=2, label='OLS fit')

# Residuals
for i in range(n_simple):
    ax.plot([X_simple[i], X_simple[i]], 
            [y_simple[i], results_simple['y_hat'][i]], 
            'gray', alpha=0.3, linewidth=0.5)

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('OLS: Minimizing Sum of Squared Residuals', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Estimated equation: y = {results_simple['beta_hat'][0]:.2f} + {results_simple['beta_hat'][1]:.2f}*X")

## 5. Comparison with Statsmodels

Let's verify our implementation matches the standard library.

In [ ]:
import statsmodels.api as sm

# Add constant to X
X_sm = sm.add_constant(X)

# Fit model
model_sm = sm.OLS(y, X_sm)
results_sm = model_sm.fit()

# Compare coefficients
print("Coefficient Comparison:")
print(f"{'Variable':<15} {'Our OLS':>12} {'Statsmodels':>12} {'Difference':>12}")
print("="*55)
for i, var in enumerate(['Intercept', 'X1', 'X2']):
    diff = results['beta_hat'][i] - results_sm.params[i]
    print(f"{var:<15} {results['beta_hat'][i]:>12.6f} {results_sm.params[i]:>12.6f} {diff:>12.2e}")

print("\nStandard Error Comparison:")
print(f"{'Variable':<15} {'Our OLS':>12} {'Statsmodels':>12} {'Difference':>12}")
print("="*55)
for i, var in enumerate(['Intercept', 'X1', 'X2']):
    diff = results['se_beta'][i] - results_sm.bse[i]
    print(f"{var:<15} {results['se_beta'][i]:>12.6f} {results_sm.bse[i]:>12.6f} {diff:>12.2e}")

print("\n✅ Our implementation matches statsmodels!")

---

## Exercises

### Exercise 1: Implement R-squared from Scratch

**Task:** Write a function that computes R-squared from residuals and the outcome variable.

**Formula:** 
$$R^2 = 1 - \frac{\sum \hat{\epsilon}_i^2}{\sum (y_i - \bar{y})^2} = 1 - \frac{SSR}{TSS}$$

**Hints:**
- SSR = Sum of Squared Residuals
- TSS = Total Sum of Squares
- Use `np.sum()` and `np.mean()`

In [ ]:
# YOUR CODE HERE
def compute_r_squared(y, residuals):
    """
    Compute R-squared.
    
    Parameters
    ----------
    y : array-like
        Actual outcome values
    residuals : array-like
        Regression residuals
    
    Returns
    -------
    r_squared : float
        R-squared value
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def compute_r_squared_solution(y, residuals):
    """Compute R-squared."""
    SSR = np.sum(residuals**2)
    TSS = np.sum((y - y.mean())**2)
    r_squared = 1 - SSR / TSS
    return r_squared

In [ ]:
# Test your function
def test_exercise_1():
    """Test R-squared computation."""
    # Test with our earlier results
    r2_computed = compute_r_squared(y, results['residuals'])
    r2_expected = results['r_squared']
    
    assert r2_computed is not None, "Function returns None - did you implement it?"
    assert np.isclose(r2_computed, r2_expected, rtol=1e-6), \
        f"R-squared incorrect: got {r2_computed:.4f}, expected {r2_expected:.4f}"
    
    print(f"✅ Correct! R-squared = {r2_computed:.4f}")
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: Confidence Intervals

**Task:** Write a function that computes 95% confidence intervals for the coefficients.

**Formula:** 
$$CI_{95\%} = \hat{\beta}_j \pm t_{0.025, n-k} \times SE(\hat{\beta}_j)$$

where $t_{0.025, n-k}$ is the 97.5th percentile of the t-distribution with $n-k$ degrees of freedom.

**Hints:**
- Use `stats.t.ppf(0.975, df)` to get the critical value
- Return lower and upper bounds as separate arrays

In [ ]:
# YOUR CODE HERE
def confidence_intervals(beta_hat, se_beta, n, k, alpha=0.05):
    """
    Compute confidence intervals for OLS coefficients.
    
    Parameters
    ----------
    beta_hat : array-like
        Coefficient estimates
    se_beta : array-like
        Standard errors
    n : int
        Sample size
    k : int
        Number of parameters (including intercept)
    alpha : float
        Significance level (default 0.05 for 95% CI)
    
    Returns
    -------
    ci_lower : array
        Lower bounds of confidence intervals
    ci_upper : array
        Upper bounds of confidence intervals
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def confidence_intervals_solution(beta_hat, se_beta, n, k, alpha=0.05):
    """Compute confidence intervals for OLS coefficients."""
    # Critical value from t-distribution
    t_crit = stats.t.ppf(1 - alpha/2, df=n-k)
    
    # Margin of error
    margin = t_crit * se_beta
    
    # Confidence intervals
    ci_lower = beta_hat - margin
    ci_upper = beta_hat + margin
    
    return ci_lower, ci_upper

In [ ]:
# Test your function
def test_exercise_2():
    """Test confidence interval computation."""
    ci_lower, ci_upper = confidence_intervals(
        results['beta_hat'], results['se_beta'], 
        results['n'], results['k']
    )
    
    assert ci_lower is not None and ci_upper is not None, \
        "Function returns None - did you implement it?"
    
    # Check that intervals contain true values (should be true ~95% of the time)
    # For this specific seed, we know they should contain true values
    contains_true = np.all((ci_lower <= beta_true) & (ci_upper >= beta_true))
    
    # Check width is reasonable
    width = ci_upper - ci_lower
    expected_width = 2 * stats.t.ppf(0.975, results['n']-results['k']) * results['se_beta']
    
    assert np.allclose(width, expected_width), \
        "Confidence interval width is incorrect"
    
    print("✅ Correct! 95% Confidence Intervals:")
    for i, var in enumerate(['Intercept', 'X1', 'X2']):
        print(f"  {var}: [{ci_lower[i]:.4f}, {ci_upper[i]:.4f}]")
    
    return True

# Uncomment to test
# test_exercise_2()

### Exercise 3: Multicollinearity Detection

**Task:** Write a function that detects multicollinearity by computing the condition number of $X'X$.

**Background:** The condition number measures how "ill-conditioned" a matrix is. A high condition number (> 30) indicates multicollinearity problems.

**Formula:**
$$\kappa(X'X) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

where $\lambda$ are eigenvalues of $X'X$.

**Hints:**
- Use `np.linalg.cond()` or compute eigenvalues with `np.linalg.eigvals()`

In [ ]:
# YOUR CODE HERE
def check_multicollinearity(X):
    """
    Check for multicollinearity using condition number.
    
    Parameters
    ----------
    X : array-like, shape (n, k)
        Design matrix (including intercept)
    
    Returns
    -------
    condition_number : float
        Condition number of X'X
    has_multicollinearity : bool
        True if condition number > 30
    """
    # TODO: Implement this function
    pass

In [ ]:
# SOLUTION (hidden)
def check_multicollinearity_solution(X):
    """Check for multicollinearity using condition number."""
    XtX = X.T @ X
    condition_number = np.linalg.cond(XtX)
    has_multicollinearity = condition_number > 30
    return condition_number, has_multicollinearity

In [ ]:
# Test your function
def test_exercise_3():
    """Test multicollinearity detection."""
    # Test with our data (should not have multicollinearity)
    cond_num, has_mult = check_multicollinearity(X_with_intercept)
    
    assert cond_num is not None, "Function returns None - did you implement it?"
    assert isinstance(has_mult, (bool, np.bool_)), "has_multicollinearity should be boolean"
    
    print(f"✅ Correct! Condition number: {cond_num:.2f}")
    print(f"   Multicollinearity detected: {has_mult}")
    
    # Create data with multicollinearity
    X_collinear = np.column_stack([np.ones(n), X1, X1 + np.random.randn(n)*0.01])
    cond_num_collinear, has_mult_collinear = check_multicollinearity(X_collinear)
    
    print(f"\n   With collinear predictors:")
    print(f"   Condition number: {cond_num_collinear:.2f}")
    print(f"   Multicollinearity detected: {has_mult_collinear}")
    
    assert has_mult_collinear, "Should detect multicollinearity in collinear data"
    print("\n✅ All multicollinearity tests passed!")
    
    return True

# Uncomment to test
# test_exercise_3()

---

## Summary

### Key Takeaways

1. **OLS in Matrix Form:** $\hat{\beta} = (X'X)^{-1}X'y$ provides a compact, generalizable formula for regression estimation.

2. **Geometric Interpretation:** OLS projects the outcome vector $y$ onto the column space of $X$, minimizing the squared distance (residuals).

3. **Numerical Stability:** Use `np.linalg.solve()` instead of computing matrix inverse explicitly for better numerical accuracy.

4. **Verification:** OLS estimates should satisfy orthogonality conditions and match standard library implementations.

5. **Foundation for Panel Methods:** Understanding OLS in matrix form is essential for fixed effects (within transformation) and random effects (GLS) estimation.

### Connections to Panel Data

- **Fixed Effects:** Apply OLS to demeaned data (within transformation removes entity effects)
- **Random Effects:** Use GLS (generalized least squares), a weighted version of OLS
- **Pooled OLS:** Direct application of OLS to panel data (ignoring panel structure)

### What's Next

In the next notebook, we'll apply these OLS fundamentals to panel data:
- Loading and structuring panel datasets
- Within and between variation decomposition  
- First look at entity effects and why pooled OLS can be biased

### Additional Practice

Try these extensions:
1. Implement robust (heteroskedasticity-consistent) standard errors
2. Add $F$-test for overall model significance
3. Compute variance inflation factors (VIF) for multicollinearity diagnosis
4. Implement weighted least squares (WLS) for heteroskedastic errors

---

**Next:** `02_data_preparation.ipynb` - Panel Data Structures and Preparation